# 03 - Embedding Models (Hue Foods RAG MVP)

Notebook này chạy **runtime thật** của Phase 3: chunk toàn bộ curated foods, load `intfloat/multilingual-e5-small` từ local cache và tạo dense embeddings thật, đồng thời fit sparse TF-IDF thật trên đủ 572 canonical chunks.

**Prerequisite**

- Chạy notebook này từ repo root hoặc từ `notebooks/`.
- Model `intfloat/multilingual-e5-small` đã có trong local Hugging Face cache. Notebook đặt `HF_HUB_OFFLINE=1` nên **không bao giờ download** hoặc gọi network; nếu cache thiếu, cell load model sẽ fail rõ ràng thay vì âm thầm tải.
- Không tốn phí: chỉ dùng model local trên CPU.

**Kết quả mong đợi khi Run All**

- Đúng 572 chunks từ curated foods Markdown.
- Dense vectors thật: 572 vectors, 384 chiều, mọi giá trị finite, norm chuẩn hóa gần 1.0.
- Query embedding thật dùng prefix `query: `, đúng 384 chiều.
- Latency embedding toàn corpus được in ra.
- Sparse TF-IDF thật: vocabulary non-empty, deterministic qua các lần fit.


In [ ]:
import sys
from pathlib import Path

for base in (Path.cwd(), *Path.cwd().parents):
    if (base / "backend").is_dir():
        sys.path.insert(0, str(base / "backend"))
        break
else:
    raise RuntimeError(
        "Khong tim thay thu muc backend/. Hay mo notebook nay tu repo root "
        "hoac tu thu muc notebooks/."
    )
print(f"backend on path: {sys.path[0]}")


## Cấu hình embedding (`settings.yaml`)

Nhóm `embedding` khai báo local baseline: provider, model ID, dimension, device, batch size và E5 instruction prefixes (`passage:` cho documents, `query:` cho queries). Nhóm `remote` khai báo OpenRouter adapter nhưng không được kích hoạt trong notebook này.


In [ ]:
from core.settings_loader import load_settings

settings = load_settings()
embedding = settings["embedding"]
print("provider:", embedding["provider"])
print("model:", embedding["model"])
print("vector_size:", embedding["vector_size"])
print("device:", embedding["device"])
print("batch_size:", embedding["batch_size"])
print("document_prefix:", repr(embedding["document_prefix"]))
print("query_prefix:", repr(embedding["query_prefix"]))


## Sparse representation - TF-IDF

`SparseEmbedder` token hóa (lowercase, bỏ ký tự không phải word, giữ Unicode tiếng Việt), fit vocabulary và document frequency theo thứ tự deterministic, rồi encode theo công thức:

```text
idf(term) = log((num_documents + 1) / (document_frequency + 1)) + 1
value = term_frequency * idf(term)
```

Cell dưới kiểm tra công thức bằng tay trên corpus nhỏ trước khi chạy trên 572 chunks thật.


In [ ]:
import math

from embedding.sparse_embedder import SparseEmbedder, tokenize

corpus = ["Bún bò Huế", "Cơm hến", "Bánh ép mè xửng", "Bún bò chay"]
sparse = SparseEmbedder().fit(corpus)

print("vocabulary size:", sparse.vocabulary_size)
print("num_documents:", sparse.num_documents)
print("tokens:", tokenize("Bún bò Huế, Cơm hến!"))
result = sparse.encode("Bún bò bò")
print("encode('Bún bò bò'):", result)
# df('bún') = df('bò') = 2 over 4 docs -> idf = log((4+1)/(2+1)) + 1
idf_bun = math.log((4 + 1) / (2 + 1)) + 1
assert result["indices"] == [sparse._vocab["bún"], sparse._vocab["bò"]]
assert math.isclose(result["values"][0], 1 * idf_bun)
assert math.isclose(result["values"][1], 2 * idf_bun)
print("hand-checked TF-IDF values: ok")


## Canonical corpus (Phase 2)

`chunk_foods_markdown()` đọc curated foods Markdown và trả đúng 572 chunks (contract đã được người dùng xác nhận). Cell dưới chạy chunking thật và in số liệu.


In [ ]:
from collections import Counter

from ingestion.chunking.markdown_chunker import chunk_foods_markdown

chunks = chunk_foods_markdown()
texts = [chunk["text"] for chunk in chunks]
print("canonical chunks:", len(texts))
print("per subcategory:", dict(Counter(c["metadata"]["subcategory"] for c in chunks)))
print("first chunk_id:", chunks[0]["metadata"]["chunk_id"])
print("last chunk_id:", chunks[-1]["metadata"]["chunk_id"])


## Dense embedding thật - local E5 từ cache

Cell dưới đặt `HF_HUB_OFFLINE=1` **trước** khi load model, rồi embed toàn bộ 572 chunk texts với prefix `passage: `. Kết quả phải là 572 vectors 384 chiều, finite và L2-normalized (norm gần 1.0). Query embedding dùng prefix `query: `. Nếu local cache thiếu model, cell này fail rõ ràng - đó là hành vi mong muốn, không có fallback.


In [ ]:
import os
import time

os.environ["HF_HUB_OFFLINE"] = "1"  # cache-only: no download, no network

import numpy as np

from embedding.embedder import SentenceTransformerEmbedder

embedder = SentenceTransformerEmbedder(
    model_id=embedding["model"],
    dimension=embedding["vector_size"],
    device=embedding["device"],
    batch_size=embedding["batch_size"],
    document_prefix=embedding["document_prefix"],
    query_prefix=embedding["query_prefix"],
)

started = time.monotonic()
vectors = embedder.embed_documents(texts)
elapsed = round(time.monotonic() - started, 1)

array = np.asarray(vectors)
norms = np.linalg.norm(array, axis=1)
query_vector = embedder.embed_query("Bún bò Huế có gì đặc biệt?")

print("model:", embedder.model_id)
print("dimension:", embedder.dimension)
print("vectors shape:", array.shape)
print("all finite:", bool(np.isfinite(array).all()))
print("norm min/mean/max:", round(float(norms.min()), 6), round(float(norms.mean()), 6), round(float(norms.max()), 6))
print("query vector dim:", len(query_vector), "| norm:", round(float(np.linalg.norm(query_vector)), 6))
print("embedding latency (572 chunks):", elapsed, "seconds")


## Sparse TF-IDF trên 572 canonical chunks

Fit `SparseEmbedder` trên toàn bộ 572 chunk texts. Vocabulary phải non-empty và cùng corpus phải tái tạo cùng kết quả encode (deterministic).


In [ ]:
from embedding.sparse_embedder import SparseEmbedder

sparse_corpus = SparseEmbedder().fit(texts)
print("vocabulary size:", sparse_corpus.vocabulary_size)
print("num_documents:", sparse_corpus.num_documents)

first = sparse_corpus.encode("bún bò huế")
second = SparseEmbedder().fit(texts).encode("bún bò huế")
print("deterministic across refits:", first == second)
print("sample indices:", first["indices"][:8])
print("sample values:", [round(v, 4) for v in first["values"][:8]])


## Checklist xác nhận Phase 3

1. `backend on path` trỏ đúng thư mục `backend/` của repo.
2. Config hiển thị `intfloat/multilingual-e5-small`, 384 dimensions, `batch_size: 64` và đúng hai prefixes.
3. Sparse sample: giá trị TF-IDF khớp tính tay; vocabulary size hợp lý.
4. Chunking thật trả đúng 572 chunks.
5. Dense E5 thật: 572 vectors 384 chiều, finite, norm gần 1.0; query embedding 384 chiều.
6. Sparse trên 572 chunks: vocabulary non-empty và deterministic.
7. Không có download, không có network, không tốn phí.
